In [1]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI
import numpy as np
import pandas as pd

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [10]:
# Initialize OpenAI API client
openai = OpenAI()

In [ ]:
# Define functions
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt + website}
    ]

def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [ ]:
# Step 1: Create your prompts

system_prompt = """
You are a smart assistant that help extract relevant data from the provided website,
the data you extracted will be summarized into pandas Dataframe based on the format available on the website table.
Respond in markdown.
"""
user_prompt = """
Here are the contents of a website.
In the content of the website you can find 3 separate tables called "BY AIRLINE", "BY ORIGIN AIRPORT", and "BY DESTINATION AIRPORT".
Extract the data from these tables and summarize them into pandas DataFrames, in linie with the format provided on the website.
"""
# Step 3: Call OpenAI
display_summary("https://www.flightaware.com/live/cancelled/")

The content provided does not include the actual data from the "BY AIRLINE", "BY ORIGIN AIRPORT", and "BY DESTINATION AIRPORT" tables. Please provide the data from these tables or the part of the website content that contains these tables so I can extract and summarize the information into pandas DataFrames for you.

In [5]:
from io import StringIO

import json
import pandas as pd
import subprocess
import sys

playwright_script = r"""
import json
from playwright.sync_api import sync_playwright

url = "https://www.flightaware.com/live/cancelled/"
with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    try:
        page = browser.new_page()
        page.goto(url, wait_until="domcontentloaded")
        page.locator("table").nth(3).wait_for()

        table_html = page.locator("table").evaluate_all(
            "(tables) => tables.slice(1, 4).map(table => table.outerHTML)"
        )
        print(json.dumps(table_html))
    finally:
        browser.close()
"""

result = subprocess.run(
    [sys.executable, "-c", playwright_script],
    check=True,
    stdout=subprocess.PIPE,
    text=True,
)
table_html = json.loads(result.stdout)

airline, origin, destination = [
    pd.read_html(StringIO(html))[0] for html in table_html
]

In [ ]:
from IPython.display import Markdown, display

tables_for_summary = [
    ("By airline", airline),
    ("By origin airport", origin),
    ("By destination airport", destination),
]

table_text = "\n\n".join(
    f"{name} (first 10 rows):\n{frame.head(10).to_string(index=False)}"
    for name, frame in tables_for_summary
)

response = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "system",
            "content": "Summarize these flight cancellation tables. Use only the supplied data; distinguish counts from percentages.",
        },
        {"role": "user", "content": table_text},
    ],
)

display(Markdown(response.choices[0].message.content))